# Domain 2: Tool Design & MCP Integration
## Task Statement 2.4: Integrate MCP Servers into Claude Code and Agent Workflows
**Certification Weight**: 20% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand how to integrate Model Context Protocol (MCP) servers into Claude Code workflows, distinguish project-level versus user-level scopes, configure secure environment variable expansion, and leverage **MCP Resources** versus **MCP Tools**.

---

## 🏗️ Architecture: MCP Server Integration Topology

```mermaid
flowchart TD
    Client["Claude Code / Agent Runtime"] --> Scopes{"Config Scope Resolver"}
    
    Scopes -->|"Project Scope (.mcp.json)"| ProjConfig["Project Config<br>(Shared team servers, repo-scoped tools)"]
    Scopes -->|"User Scope (~/.claude/mcp.json)"| UserConfig["User Global Config<br>(Personal developer tools, local proxies)"]
    
    ProjConfig --> EnvExpand["Environment Variable Expansion<br>API_KEY: ${GITHUB_TOKEN}"]
    UserConfig --> EnvExpand
    
    EnvExpand --> MCPLaunch["Launch MCP Server Process (Stdio / SSE)"]
    
    MCPLaunch --> Capabilities{"Server Capabilities"}
    Capabilities -->|"Read-only Reference"| Resources["MCP Resources<br>(Direct URI access: db://schema, doc://api)"]
    Capabilities -->|"Executable Functions"| Tools["MCP Tools<br>(Parameterized actions: run_query, create_pr)"]
```

---

## ⚠️ Critical Exam Distinctions & Anti-Patterns

| Configuration / Feature | Correct MCP Implementation | Exam Trap to Reject |
|---|---|---|
| **Credential Management** | Use `${VAR_NAME}` syntax to interpolate secrets from the host environment. | **TRAP**: Hardcoding plain-text API tokens or database passwords in checked-in configuration files. |
| **Config Scoping** | Use **Project scope** (`.mcp.json`) for team-shared repo tools; use **User scope** for personal workstation utilities. | **TRAP**: Putting team project dependencies in user home directories where teammates cannot access them. |
| **Resources vs Tools** | Expose static schemas and documentation as **MCP Resources** (read directly without tool calls). | **TRAP**: Wrapping static read-only reference data in dynamic executable tools that consume multiple round-trips. |

---

## 💡 Architectural Decision Matrix

| Capability | Primitives | Best Use Case | Performance Benefit |
|---|---|---|---|
| **MCP Resources** | Static URIs (`resource://...`) | Database schemas, design system specs, API docs | Injected directly into context without tool execution latency. |
| **MCP Tools** | JSON Schema functions | Queries with dynamic parameters, write operations | Invoked selectively only when required. |
| **MCP Prompts** | Reusable prompt templates | Standardized bug report triage, PR review checklists | Consistency across developer teams. |

---

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Setup
import json
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 2.4 MCP Server Integration.")

### 📦 Step 2: Simulated MCP Resource Catalog & Safe Edit Function
We implement an MCP service exposing resources and a safe file editing function with fallback detection.

In [ ]:
# Step 2: MCP Service & Safe Edit Implementation
class DeveloperMCPService:
    """Simulates an MCP service offering read-only Resources and operational Tools."""
    RESOURCES = {
        "catalog://schema/orders": {
            "uri": "catalog://schema/orders",
            "name": "Orders Database Schema",
            "mimeType": "application/json",
            "text": json.dumps({
                "tables": {
                    "orders": ["id", "customer_id", "total_cents", "status", "created_at"],
                    "order_items": ["id", "order_id", "sku", "quantity", "unit_price_cents"],
                }
            }),
        },
        "catalog://docs/api_endpoints": {
            "uri": "catalog://docs/api_endpoints",
            "name": "API Route Catalog",
            "mimeType": "text/markdown",
            "text": "# Routes\n- GET /api/v1/orders/{id}\n- POST /api/v1/orders/{id}/refund",
        },
    }

    @classmethod
    def read_resource(cls, uri: str) -> Optional[Dict[str, Any]]:
        return cls.RESOURCES.get(uri)

    @classmethod
    def list_resources(cls) -> List[Dict[str, str]]:
        return [{"uri": v["uri"], "name": v["name"]} for v in cls.RESOURCES.values()]


def safe_edit_file(file_content: str, old_string: str, new_string: str) -> Tuple[bool, str]:
    """
    Safely edits file content. Detects non-unique anchors and triggers fallback to Read+Write.
    """
    match_count = file_content.count(old_string)
    if match_count == 0:
        return False, "Anchor string not found. Triggering Read + Write fallback."
    if match_count > 1:
        return False, f"Ambiguous edit: Anchor string found {match_count} times. Triggering Read + Write fallback."
    
    updated_content = file_content.replace(old_string, new_string, 1)
    return True, updated_content

print(" DeveloperMCPService and safe_edit_file ready.")

### 🧪 Step 3: Interactive Simulation
1. Read catalog schema resource (`catalog://schema/orders`).
2. Perform unique surgical edit -> Success.
3. Attempt ambiguous edit (multiple matches) -> Triggers fallback.

In [ ]:
# Step 3: Simulation
# 1. Read MCP Resource
schema_res = DeveloperMCPService.read_resource("catalog://schema/orders")
print("MCP Resource Read Result:", json.loads(schema_res["text"]))

# 2. Successful Unique Edit
code_sample = "def calculate_tax(amount):\n    rate = 0.05\n    return amount * rate\n"
ok, new_code = safe_edit_file(code_sample, "rate = 0.05", "rate = 0.08")
print("\nUnique Edit Status:", ok)
print("Updated Code:\n", new_code)

# 3. Ambiguous Edit (fails unique match)
ambiguous_code = "x = 1\nx = 1\n"
ok_amb, msg = safe_edit_file(ambiguous_code, "x = 1", "x = 2")
print("\nAmbiguous Edit Status:", ok_amb)
print("Fallback Message:", msg)

### 🔬 Step 4: Verification Assertions
We assert that:
1. MCP Resources can be read by URI without tool execution.
2. Unique strings are safely replaced.
3. Ambiguous anchors cleanly reject with a Read+Write fallback notice.

In [ ]:
# Step 4: Verification Checks
assert schema_res is not None
assert "orders" in schema_res["text"]

assert ok is True
assert "rate = 0.08" in new_code

assert ok_amb is False
assert "Triggering Read + Write fallback" in msg

print(" All verification assertions passed! MCP integration validated.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Expose static database schemas and documentation as **MCP Resources**, reserving Tools for active queries and actions.
- **Rule 2**: Surgical file edits require unique anchor matching.
- **Rule 3**: If an anchor match is ambiguous, always fall back to a full **Read + Write** workflow.